In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Określanie opcji Samplera

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';







<Accordion>
<AccordionItem title="Wersje pakietów">

Kod na tej stronie został opracowany z użyciem poniższych wymagań.
Zalecamy używanie tych wersji lub nowszych.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Możesz używać opcji do dostosowania prymitywu Sampler. Ta sekcja skupia się na tym, jak określać opcje prymitywów Qiskit Runtime. Chociaż interfejs metody `run()` prymitywów jest wspólny dla wszystkich implementacji, ich opcje już nie są. Zapoznaj się z odpowiednimi referencjami API, aby uzyskać informacje o opcjach [`qiskit.primitives.BackendSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendSamplerV2) i [`qiskit_aer.primitives.SamplerV2`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.primitives.SamplerV2.html).

<span id="pass-options"></span>
## Ustawianie opcji Samplera
Opcje możesz ustawiać podczas inicjalizacji Samplera, po jego inicjalizacji lub aktualizować je po inicjalizacji. Aby zapoznać się z instrukcjami dotyczącymi tych technik, zobacz temat [Wprowadzenie do opcji](/guides/runtime-options-overview#options-precedence).

Dodatkowo możesz ustawiać wartość `shots` w metodzie `run()`, co zostało opisane w poniższej sekcji.
<span id="run-method"></span>
### Metoda Run()
Jedyne wartości, które możesz przekazać do `run()`, to te zdefiniowane w interfejsie, czyli `shots`. Nadpisuje to wartość ustawioną dla `default_shots` dla bieżącego uruchomienia.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)

sampler = Sampler(mode=backend)
# Default shots to use if not specified in run()
sampler.options.default_shots = 500
# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d8286680bvlc73d1vmu0', 'sampler')>

### Special cases

<span id="shots"></span>
#### Shots

The `SamplerV2.run` method accepts two arguments: a list of PUBs, each of which can specify a PUB-specific value for shots, and a shots keyword argument. These shot values are a part of the Sampler execution interface, and are independent of the Runtime Sampler's options.  They take precedence over any values specified as options in order to comply with the Sampler abstraction.

However, if `shots` is not specified by any PUB or in the run keyword argument (or if they are all `None`), then the shots value from the options is used, most notably `default_shots`.

To summarize, this is the order of precedence for specifying shots in the Sampler, for any particular PUB:

1. If the PUB specifies shots, use that value.
2. If the `shots` keyword argument is specified in `run`, use that value.
4. If `twirling` is enabled  (True by default), then the product of `num_randomizations` and `shots_per_randomization`, as specified as  [`twirling` options](/docs/api/qiskit-ibm-runtime/options-twirling-options), is used.
5. If `sampler.options.default_shots` is specified, use that value.

Thus, if shots are specified in all possible places, the one with highest precedence (shots specified in the PUB) is used.

Example:

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)


# Setting shots during primitive initialization
sampler = Sampler(mode=backend, options={"default_shots": 4096})

# Setting options after primitive initialization
# This uses auto-complete.
sampler.options.default_shots = 2000

# This does bulk update.  The value for default_shots is overridden if you specify shots with run() or in the PUB.
sampler.options.update(
    default_shots=1024, dynamical_decoupling={"sequence_type": "XpXm"}
)

# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d82868ugbeec73alfa80', 'sampler')>

### Przypadki szczególne
<span id="shots"></span>
#### Shots

Metoda `SamplerV2.run` przyjmuje dwa argumenty: listę PUBów, z których każdy może określać PUB-specyficzną wartość shots, oraz argument kluczowy shots. Te wartości shots są częścią interfejsu wykonania Samplera i są niezależne od opcji Samplera Runtime. Mają one pierwszeństwo przed wartościami określonymi jako opcje, aby zachować zgodność z abstrakcją Samplera.

Jednak jeśli `shots` nie jest określone przez żaden PUB ani w argumencie kluczowym run (lub jeśli wszystkie są `None`), używana jest wartość shots z opcji, przede wszystkim `default_shots`.

Podsumowując, oto kolejność pierwszeństwa dla określania shots w Samplerze, dla danego PUBa:

1. Jeśli PUB określa shots, użyj tej wartości.
2. Jeśli argument kluczowy `shots` jest określony w `run`, użyj tej wartości.
4. Jeśli `twirling` jest włączony (domyślnie True), używany jest iloczyn `num_randomizations` i `shots_per_randomization`, określony jako [opcje `twirling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options).
5. Jeśli `sampler.options.default_shots` jest określone, użyj tej wartości.

Zatem jeśli shots są określone we wszystkich możliwych miejscach, używane jest to z najwyższym pierwszeństwem (shots określone w PUBie).

Przykład: